In [1]:
from __future__ import print_function, division
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns',None)
np.set_printoptions(threshold=np.inf)

In [ ]:
# read dataset
DATA_DIR = r'D:\data'
df = pd.read_csv(r'D:\data\dataset.csv')
pd.set_option('display.max_columns',None)
training_sequence = df['Sequence'].to_list()
print(training_sequence)

['RG', 'HG', 'NG', 'QG', 'FW', 'NY', 'FC', 'KT', 'PH', 'LR', 'QW', 'CK', 'AQ', 'MY', 'NI', 'EH', 'RS', 'HK', 'WY', 'RS', 'GG', 'KG', 'DG', 'EG', 'SG', 'TG', 'CG', 'WG', 'PG', 'AG', 'VG', 'IG', 'LG', 'MG', 'FG', 'YG', 'WA', 'SV', 'MI', 'DF', 'EA', 'YV', 'DP', 'GH', 'FF', 'AYG', 'TDG', 'TKG', 'FRG', 'GKG', 'GYG', 'SYG', 'TYG', 'QWI', 'GPM', 'QPF', 'STR', 'YQG', 'HCG', 'VMG', 'ILG', 'QYG', 'DHG', 'WNG', 'ERG', 'CLG', 'YHG', 'FTG', 'YQG', 'RVG', 'MAG', 'LDG', 'NFG', 'HRG', 'VWG', 'PSG', 'AIG', 'EHG', 'WIC', 'FGQ', 'LPV', 'YTC', 'RSM', 'HKV', 'NML', 'KET', 'VFA', 'HCY', 'YYL', 'DCL', 'ESN', 'VQI', 'IQM', 'HYR', 'NKF', 'WYL', 'RVE', 'GTF', 'TDGG', 'TGKG', 'SEPG', 'AYKG', 'FGYG', 'FKFG', 'FGRG', 'FGKG', 'YAKG', 'GMPN', 'YGTK', 'GTID', 'MPNS', 'ACAY', 'GGTD', 'GTAK', 'SYFG', 'GFAA', 'AACA', 'GPTL', 'EGMP', 'RPFW', 'IGPS', 'FPGM', 'KGDY', 'FQAY', 'GTFI', 'TDDA', 'THYK', 'DDAD', 'PGTF', 'TFIL', 'FCGG', 'AYRG', 'TDPG', 'SEGG', 'MSPG', 'QACG', 'TDDAG', 'MNSFG', 'SEMNG', 'FCGYG', 'FAYRG', 'KKKKG', 

In [3]:
from FeatureExtraction import extract_features
X_train = extract_features(training_sequence)

(306, 20) (306, 400) (306, 280) (306, 16)


In [4]:
df2 = pd.DataFrame(X_train)
df2_non_zero = df2.loc[:, (df2 != 0).any(axis=0)]
removed_indices = df2.columns[~df2.columns.isin(df2_non_zero.columns)].tolist()
X_filtered = np.array(df2_non_zero)

In [5]:
# process train y
y_train1 = df['Score'].to_numpy(dtype=float)

count1 = (df['Score'] == 1).sum()
count0 = (df['Score'] == 0).sum()

print(count1, count0)

174 132


In [6]:
# GaussianNB
from sklearn.naive_bayes import GaussianNB

def nb_classifier():
   return GaussianNB()

In [7]:
# KNeighborsClassifier
from sklearn.neighbors import KNeighborsClassifier

def knn_classifier():
    return KNeighborsClassifier(n_neighbors=5, p=1, weights='distance')  

In [8]:
# GradientBoostingClassifier
from sklearn.ensemble import GradientBoostingClassifier

def modelGB():
    return GradientBoostingClassifier(n_estimators=100, learning_rate=0.5, max_depth=7, random_state=42)

In [9]:
# RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier

def modelRF():
    return RandomForestClassifier(random_state=42)

In [10]:
# SVC
from sklearn.svm import SVC

def modelSVC():
    return SVC(random_state=42)

In [11]:
# LogisticRegression
from sklearn.linear_model import LogisticRegression

def modelLR():
    return LogisticRegression()

In [12]:
# XGB
import xgboost as xgb

def xgb_clf():
    return xgb.XGBClassifier(colsample_bytree=1.0, learning_rate=0.1, max_depth=3, n_estimators=100, subsample=1.0, eval_metric='logloss', random_state=42)

In [13]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

def adaboost_clf():
    base_model = DecisionTreeClassifier(max_depth=1, random_state=42)
    return AdaBoostClassifier(base_estimator=base_model, n_estimators=200, random_state=42)

In [14]:
from sklearn.ensemble import VotingClassifier

def voting_clf1():
    clf1 = GaussianNB()
    clf2 = RandomForestClassifier(random_state=42)
    clf3 = SVC(random_state=42, probability=True)
    clf4 = xgb.XGBClassifier(colsample_bytree=1.0, learning_rate=0.1, max_depth=3, n_estimators=100, subsample=1.0, eval_metric='logloss', random_state=42)
    return VotingClassifier(estimators=[('lr', clf1), ('dt', clf2), ('svc', clf3), ('xgb', clf4)], voting='soft')

In [15]:
def voting_clf2():
    clf1 = RandomForestClassifier(random_state=42)
    clf2 = SVC(random_state=42, probability=True)
    return VotingClassifier(estimators=[('rf', clf1), ('svm', clf2)], voting='soft') 

In [16]:
def voting_clf3():
    clf1 = GaussianNB()
    clf2 = RandomForestClassifier(random_state=42)
    clf3 = SVC(random_state=42, probability=True)
    return VotingClassifier(estimators=[('lr', clf1), ('dt', clf2), ('svc', clf3)], voting='soft')

In [17]:
from sklearn.ensemble import StackingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

def stack():
    stack1 = GaussianNB()
    stack2 = RandomForestClassifier( random_state=42)
    stack3 = SVC(random_state=42, probability=True)
    return StackingClassifier(estimators=[('lr', stack1), ('RF', stack2), ('SVC', stack3)])

In [18]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler

def kfold_evaluate_model(base_model, X, y):

    # 定义 k 折交叉验证
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)

    accuracies = []

    for train_index, test_index in kfold.split(X, y):

        x_train, x_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        
        scaler = MinMaxScaler(feature_range=(0, 1))
        x_train_scaled = scaler.fit_transform(x_train)
        x_test_scaled = scaler.transform(x_test)

        model = base_model()
        model.fit(x_train_scaled, y_train)
        y_pred = model.predict(x_test_scaled)
        accuracies.append(accuracy_score(y_test, y_pred))

    mean_accuracy = np.mean(accuracies)
    std_accuracy = np.std(accuracies)  
    return mean_accuracy, std_accuracy

In [23]:
tol = 1e-3
patience = 5

def feature_selection(X, y, base_model):
    
    name = base_model().__class__.__name__

    all_list = list(range(len(X[0])))  # Indices of all features

    remain_list = []  # List of selected feature indices
    max_iterations = 40  # Set maximum iterations

    best_score = -np.inf
    max_list = []

    no_improvement_count = 0

    for i in range(max_iterations):
        
        all_r2 = []

        for indice in all_list:
            
            # Display current progress
            print(f"{i}/{max_iterations} {indice}/{len(all_list)}", end='\r')  

            # Skip if feature is already selected
            if indice in remain_list:  
                all_r2.append(-np.inf)
                continue
            
            # Add the current feature to the candidate list
            temp_remain_list = remain_list + [indice]  
            
            # Use the selected feature subset
            X_new = X[:, temp_remain_list]  

            try:
                # Cross-validation with scaling
                mean_acc, _ = kfold_evaluate_model(base_model, X_new, y)
            except:
                mean_acc = -np.inf  

            all_r2.append(mean_acc)

        max_id = np.argmax(all_r2)  # Index of the feature with the maximum accuracy
        remain_list.append(max_id)  # Add that feature to the selected list

        score = np.max(all_r2)

        if score - best_score < tol:
            no_improvement_count += 1
        else:
            no_improvement_count = 0

        # Print the maximum accuracy and selected features
        if best_score < score:
            best_score = score
            max_list = remain_list.copy()

        print(best_score, max_list, no_improvement_count)

        if no_improvement_count >= patience:
            print(f"Early stopping at iteration {i + 1}, no improvement.")
            break
    
    X_new = X[:, max_list]
    mean_acc, std_acc = kfold_evaluate_model(base_model, X_new, y)
    print(name, mean_acc, std_acc, max_list)
    
    return all_r2, remain_list

all_r2, y_in_removed_lists = feature_selection(X_filtered, y_train1, modelSVC) 

0.6077736647276573 [28] 0
0.6405605499735589 [28, 32] 0
0.6635642517186674 [28, 32, 0] 0
0.6865150713907986 [28, 32, 0, 24] 0
0.7060285563194078 [28, 32, 0, 24, 21] 0
0.7255949233209942 [28, 32, 0, 24, 21, 18] 0
0.7452141723955579 [28, 32, 0, 24, 21, 18, 22] 0
0.7517715494447382 [28, 32, 0, 24, 21, 18, 22, 166] 0
0.7615018508725543 [28, 32, 0, 24, 21, 18, 22, 166, 33] 0
0.7680592279217346 [28, 32, 0, 24, 21, 18, 22, 166, 33, 325] 0
0.7746166049709149 [28, 32, 0, 24, 21, 18, 22, 166, 33, 325, 79] 0
0.7779481755684823 [28, 32, 0, 24, 21, 18, 22, 166, 33, 325, 79, 316] 0
0.7812268640930725 [28, 32, 0, 24, 21, 18, 22, 166, 33, 325, 79, 316, 39] 0
0.7845055526176626 [28, 32, 0, 24, 21, 18, 22, 166, 33, 325, 79, 316, 39, 385] 0
0.7877313590692756 [28, 32, 0, 24, 21, 18, 22, 166, 33, 325, 79, 316, 39, 385, 396] 0
0.7910100475938657 [28, 32, 0, 24, 21, 18, 22, 166, 33, 325, 79, 316, 39, 385, 396, 47] 0
0.7943416181914331 [28, 32, 0, 24, 21, 18, 22, 166, 33, 325, 79, 316, 39, 385, 396, 47, 17] 